# V35: controlled host prompt sampling
Two matched Qwen3-4B arms, two fixed folds. SupCon paper https://arxiv.org/abs/2004.11362 and author implementation https://github.com/HobbitLong/SupContrast. Crossed prompt sampling is our untested extension. No automatic submission; see frozen protocol.

In [ ]:
import subprocess,sys,importlib.metadata
from packaging.version import Version
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)

import glob,hashlib
from pathlib import Path
wheels=glob.glob('/kaggle/input/**/bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl',recursive=True)
assert len(wheels)==1,wheels
assert hashlib.sha256(Path(wheels[0]).read_bytes()).hexdigest()=='173d137610468bec9cddbaa2e049254e97792657ab984e3e737bec1772c1668c'
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall',wheels[0]],check=True)
assert importlib.metadata.version('bitsandbytes')=='0.50.0'


In [ ]:
import os, sys, subprocess, json, hashlib, time, signal
from pathlib import Path
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[key]='2'
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',TOKENIZERS_PARALLELISM='false')
Path('run_confirmation_arm_v35.py').write_text('"""Run one fixed recipe on independent host folds; each fit resumes external-only."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom prepare_likelihood_v33 import prepare\n\n\ndef main():\n    root = Path.cwd()\n    plan = json.loads((root/\'plan.json\').read_text())\n    arm = sys.argv[1]\n    spec = plan[\'arms\'][arm]\n    records = []\n    for fold in plan[\'protocol\'][\'primary_folds\'][\'train\']:\n        folder = root/arm/f\'fold-{fold}\'\n        assert not folder.exists(), \'Never overwrite an existing fold run\'\n        folder.mkdir(parents=True)\n        params = dict(spec[\'parameters\'])\n        params.update(experiment=\'improvement-v35\', pilot_fold=fold,\n                      labels=plan[\'labels\'], model_dir=spec[\'model_dir\'],\n                      checkpoint=spec[\'checkpoint\'], checkpoint_sha256=spec[\'checkpoint_sha256\'],\n                      confirmation_split=f\'primary-{fold}\', confirmation_fold=fold,\n                      host_sha256=plan[\'host_sha256\'], **plan[\'fold_sizes\'][str(fold)])\n        prepare(plan[\'host_path\'], plan[\'fold_by_prompt\'], params, folder)\n        (folder/\'training-plan.json\').write_text(json.dumps(params,indent=2))\n        (folder/\'plan.json\').write_text(json.dumps(params,indent=2))\n        worker = root/spec[\'training_worker\']\n        env = dict(os.environ,V18_VARIANT=spec[\'variant\'])\n        def run(path, environment):\n            remaining = plan[\'deadline_unix\']-time.time()\n            if remaining <= 0:\n                raise TimeoutError(\'Confirmation notebook wall budget exhausted\')\n            subprocess.run([sys.executable,\'-u\',str(path)],cwd=folder,env=environment,\n                           check=True,timeout=remaining)\n        run(worker,env)\n        training = json.loads((folder/spec[\'variant\']/\'result.json\').read_text())\n        assert training[\'completed\'] and training[\'validation_rows\']==params[\'validation_rows\']\n        readout = dict(params)\n        readout.update(spec[\'readout\'])\n        readout.update(host_path=plan[\'host_path\'],upstream_path=str(folder))\n        weight = folder/spec[\'variant\']/\'final/adapter/adapter_model.safetensors\'\n        readout[\'adapter_sha256\'] = {spec[\'readout_variant\']:hashlib.sha256(weight.read_bytes()).hexdigest()}\n        (folder/\'readout-plan.json\').write_text(json.dumps(readout,indent=2))\n        (folder/\'plan.json\').write_text(json.dumps(readout,indent=2))\n        try:\n            run(root/spec[\'readout_worker\'],dict(env,V19_VARIANTS=json.dumps([spec[\'readout_variant\']])))\n        finally:\n            (folder/\'plan.json\').write_text(json.dumps(params,indent=2))\n        result = json.loads((folder/(spec[\'readout_variant\']+\'_result.json\')).read_text())\n        record = {\'arm\':arm,\'fold\':fold,\'training\':training,\'readout\':result}\n        records.append(record)\n        with (root/(arm+\'-progress.jsonl\')).open(\'a\') as f:\n            f.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record),flush=True)\n    (root/(arm+\'-complete.json\')).write_text(json.dumps({\'completed\':True,\'folds\':records},indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n')
Path('prepare_likelihood_v33.py').write_text('"""Prepare one fixed host holdout, retaining related-prompt group boundaries."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\n\ndef prepare(host_path, split_by_prompt, plan, output):\n    host_path = Path(host_path)\n    assert hashlib.sha256(host_path.read_bytes()).hexdigest() == plan[\'host_sha256\']\n    host = pd.read_csv(host_path, dtype={\'id\': str, \'prompt_id\': str})\n    assert len(host) == 2699 and host.id.is_unique\n    assert set(host.prompt_id) == set(split_by_prompt)\n    labels = sorted(host.model.unique())\n    assert labels == plan[\'labels\'] and len(labels) == 18\n    host[\'group\'] = host.prompt_id.map(lambda p: str(split_by_prompt[p][\'group\']))\n    assigned = host.prompt_id.map(lambda p: int(split_by_prompt[p][\'fold\']))\n    assert host.assign(fold=assigned).groupby(\'group\').fold.nunique().max() == 1\n    heldout = assigned == plan[\'confirmation_fold\']\n    host[\'stage\'] = \'host\'\n    host[\'label\'] = host.model.map({name: i for i, name in enumerate(labels)}).astype(int)\n    train = host.loc[~heldout].reset_index(drop=True)\n    valid = host.loc[heldout].reset_index(drop=True)\n    assert len(train) == plan[\'training_rows\'] and len(valid) == plan[\'validation_rows\']\n    assert set(train.group).isdisjoint(valid.group)\n    assert set(train.prompt_id).isdisjoint(valid.prompt_id)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    paths = [output / name for name in [\'training_frame.parquet\', \'validation_frame.parquet\']]\n    assert not any(p.exists() for p in paths), \'Existing prepared frames must not be overwritten\'\n    for path, frame in zip(paths, [train, valid]):\n        frame.to_parquet(path, index=False)\n    audit = {\'split\': plan[\'confirmation_split\'], \'training_rows\': len(train),\n             \'validation_rows\': len(valid), \'training_groups\': train.group.nunique(),\n             \'validation_groups\': valid.group.nunique(), \'labels\': labels,\n             \'sha256\': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}}\n    (output / \'data-audit.json\').write_text(json.dumps(audit, indent=2))\n    return audit\n')
Path('sampling_v35.py').write_text('"""Host-only crossed-author/prompt batches; V35 extension, not original SupCon."""\nimport numpy as np\n\n\ndef crossed_batches(frame, steps, seed):\n    if set(frame.stage) != {\'host\'}:\n        raise ValueError(\'Crossed sampling requires verified host prompt IDs\')\n    if not frame.index.is_unique or frame.duplicated([\'label\', \'group\']).any():\n        raise ValueError(\'Expected exactly one response per author/prompt cell\')\n    groups = {label: {p: int(sub.index[0]) for p, sub in group.groupby(\'group\')}\n              for label, group in frame.groupby(\'label\')}\n    labels = sorted(groups)\n    if len(labels) < 2 or any(len(set(groups[a]) & set(groups[b])) < 2\n                            for a in labels for b in labels if a != b):\n        raise ValueError(\'Every author pair needs two shared prompt IDs\')\n    rng = np.random.default_rng(seed)\n    for _ in range(steps):\n        a, b = rng.choice(labels, 2, replace=False)\n        shared = sorted(set(groups[a]) & set(groups[b]))\n        p, q = rng.choice(shared, 2, replace=False)\n        yield [groups[a][p], groups[a][q], groups[b][p], groups[b][q]]\n')
Path('independent_worker.py').write_text('"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n                                            local_files_only=True, attn_implementation="sdpa")\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan).cuda()\n    # Restore only an independently verified external-only checkpoint.\n    import hashlib\n    from safetensors.torch import load_file\n    from peft import set_peft_model_state_dict\n    checkpoint = Path(plan[\'checkpoint\'])\n    for relative,digest in plan[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\n    state = load_file(str(checkpoint/\'adapter/adapter_model.safetensors\'))\n    restored = set_peft_model_state_dict(model.encoder,state)\n    assert not restored.unexpected_keys, restored.unexpected_keys\n    assert not [k for k in restored.missing_keys if \'lora_\' in k], restored.missing_keys\n    heads = torch.load(checkpoint/\'heads.pt\',map_location=\'cpu\',weights_only=True)\n    for name in [\'norm\',\'head\',\'projection\']:\n        getattr(model,name).load_state_dict(heads[name],strict=True)\n    assert set(frame.stage)=={\'host\'}\n    assert len(frame)==plan[\'training_rows\'] and len(valid)==plan[\'validation_rows\']\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\n    log(event=\'external_checkpoint_loaded\',sha256=plan[\'checkpoint_sha256\'],\n        fit_rows=len(frame),validation_rows=len(valid))\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda")\n    skipped = 0\n    for stage, epochs in [("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('independent_readout.py').write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer\nfrom peft import PeftModel\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n                                         attn_implementation="sdpa", local_files_only=True)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.cuda().eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
Path('crossed_worker.py').write_text('"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nfrom sampling_v35 import crossed_batches as paired_batches\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n                                            local_files_only=True, attn_implementation="sdpa")\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan).cuda()\n    # Restore only an independently verified external-only checkpoint.\n    import hashlib\n    from safetensors.torch import load_file\n    from peft import set_peft_model_state_dict\n    checkpoint = Path(plan[\'checkpoint\'])\n    for relative,digest in plan[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\n    state = load_file(str(checkpoint/\'adapter/adapter_model.safetensors\'))\n    restored = set_peft_model_state_dict(model.encoder,state)\n    assert not restored.unexpected_keys, restored.unexpected_keys\n    assert not [k for k in restored.missing_keys if \'lora_\' in k], restored.missing_keys\n    heads = torch.load(checkpoint/\'heads.pt\',map_location=\'cpu\',weights_only=True)\n    for name in [\'norm\',\'head\',\'projection\']:\n        getattr(model,name).load_state_dict(heads[name],strict=True)\n    assert set(frame.stage)=={\'host\'}\n    assert len(frame)==plan[\'training_rows\'] and len(valid)==plan[\'validation_rows\']\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\n    log(event=\'external_checkpoint_loaded\',sha256=plan[\'checkpoint_sha256\'],\n        fit_rows=len(frame),validation_rows=len(valid))\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda")\n    skipped = 0\n    for stage, epochs in [("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('crossed_readout.py').write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer\nfrom peft import PeftModel\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n                                         attn_implementation="sdpa", local_files_only=True)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.cuda().eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
plan=json.loads("{\"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"], \"fold_by_prompt\": {\"p000\": {\"group\": \"host:p000\", \"fold\": 1}, \"p001\": {\"group\": \"host:p001\", \"fold\": 4}, \"p002\": {\"group\": \"host:p002\", \"fold\": 3}, \"p003\": {\"group\": \"host:p003\", \"fold\": 3}, \"p004\": {\"group\": \"host:p004\", \"fold\": 1}, \"p005\": {\"group\": \"host:p005\", \"fold\": 2}, \"p006\": {\"group\": \"host:p006\", \"fold\": 3}, \"p007\": {\"group\": \"host:p007\", \"fold\": 2}, \"p008\": {\"group\": \"host:p008\", \"fold\": 3}, \"p009\": {\"group\": \"host:p009\", \"fold\": 0}, \"p010\": {\"group\": \"host:p010\", \"fold\": 1}, \"p011\": {\"group\": \"host:p011\", \"fold\": 1}, \"p012\": {\"group\": \"host:p012\", \"fold\": 0}, \"p013\": {\"group\": \"host:p013\", \"fold\": 3}, \"p014\": {\"group\": \"host:p014\", \"fold\": 4}, \"p015\": {\"group\": \"host:p015\", \"fold\": 1}, \"p016\": {\"group\": \"host:p016\", \"fold\": 1}, \"p017\": {\"group\": \"host:p017\", \"fold\": 3}, \"p018\": {\"group\": \"host:p018\", \"fold\": 0}, \"p019\": {\"group\": \"host:p019\", \"fold\": 0}, \"p020\": {\"group\": \"host:p020\", \"fold\": 4}, \"p021\": {\"group\": \"host:p021\", \"fold\": 4}, \"p022\": {\"group\": \"host:p022\", \"fold\": 1}, \"p023\": {\"group\": \"host:p023\", \"fold\": 2}, \"p024\": {\"group\": \"host:p024\", \"fold\": 2}, \"p025\": {\"group\": \"host:p025\", \"fold\": 2}, \"p026\": {\"group\": \"host:p026\", \"fold\": 0}, \"p027\": {\"group\": \"host:p027\", \"fold\": 1}, \"p028\": {\"group\": \"host:p028\", \"fold\": 1}, \"p029\": {\"group\": \"host:p029\", \"fold\": 0}, \"p030\": {\"group\": \"host:p030\", \"fold\": 0}, \"p031\": {\"group\": \"host:p031\", \"fold\": 0}, \"p032\": {\"group\": \"host:p032\", \"fold\": 1}, \"p033\": {\"group\": \"host:p033\", \"fold\": 2}, \"p034\": {\"group\": \"host:p034\", \"fold\": 2}, \"p035\": {\"group\": \"host:p035\", \"fold\": 2}, \"p036\": {\"group\": \"host:p036\", \"fold\": 0}, \"p037\": {\"group\": \"host:p037\", \"fold\": 4}, \"p038\": {\"group\": \"host:p038\", \"fold\": 3}, \"p039\": {\"group\": \"host:p039\", \"fold\": 2}, \"p040\": {\"group\": \"host:p040\", \"fold\": 1}, \"p041\": {\"group\": \"host:p041\", \"fold\": 4}, \"p042\": {\"group\": \"host:p042\", \"fold\": 1}, \"p043\": {\"group\": \"host:p043\", \"fold\": 2}, \"p044\": {\"group\": \"host:p044\", \"fold\": 1}, \"p045\": {\"group\": \"host:p045\", \"fold\": 0}, \"p046\": {\"group\": \"host:p046\", \"fold\": 3}, \"p047\": {\"group\": \"host:p047\", \"fold\": 2}, \"p048\": {\"group\": \"host:p048\", \"fold\": 4}, \"p049\": {\"group\": \"host:p049\", \"fold\": 2}, \"p050\": {\"group\": \"host:p050\", \"fold\": 3}, \"p051\": {\"group\": \"host:p051\", \"fold\": 1}, \"p052\": {\"group\": \"host:p052\", \"fold\": 4}, \"p053\": {\"group\": \"host:p053\", \"fold\": 2}, \"p054\": {\"group\": \"host:p054\", \"fold\": 3}, \"p055\": {\"group\": \"host:p055\", \"fold\": 0}, \"p056\": {\"group\": \"host:p056\", \"fold\": 0}, \"p057\": {\"group\": \"host:p057\", \"fold\": 4}, \"p058\": {\"group\": \"host:p058\", \"fold\": 4}, \"p059\": {\"group\": \"host:p059\", \"fold\": 3}, \"p060\": {\"group\": \"host:p060\", \"fold\": 1}, \"p061\": {\"group\": \"host:p061\", \"fold\": 3}, \"p062\": {\"group\": \"host:p062\", \"fold\": 2}, \"p063\": {\"group\": \"host:p063\", \"fold\": 3}, \"p064\": {\"group\": \"host:p064\", \"fold\": 0}, \"p065\": {\"group\": \"host:p065\", \"fold\": 1}, \"p066\": {\"group\": \"host:p066\", \"fold\": 1}, \"p067\": {\"group\": \"host:p067\", \"fold\": 1}, \"p068\": {\"group\": \"host:p068\", \"fold\": 0}, \"p069\": {\"group\": \"host:p069\", \"fold\": 0}, \"p070\": {\"group\": \"host:p070\", \"fold\": 2}, \"p071\": {\"group\": \"host:p071\", \"fold\": 4}, \"p072\": {\"group\": \"host:p072\", \"fold\": 3}, \"p073\": {\"group\": \"host:p073\", \"fold\": 0}, \"p074\": {\"group\": \"host:p074\", \"fold\": 4}, \"p075\": {\"group\": \"host:p075\", \"fold\": 1}, \"p076\": {\"group\": \"host:p076\", \"fold\": 0}, \"p077\": {\"group\": \"host:p077\", \"fold\": 2}, \"p078\": {\"group\": \"host:p078\", \"fold\": 0}, \"p079\": {\"group\": \"host:p079\", \"fold\": 3}, \"p080\": {\"group\": \"host:p080\", \"fold\": 2}, \"p081\": {\"group\": \"host:p081\", \"fold\": 1}, \"p082\": {\"group\": \"host:p082\", \"fold\": 0}, \"p083\": {\"group\": \"host:p083\", \"fold\": 3}, \"p084\": {\"group\": \"host:p084\", \"fold\": 2}, \"p085\": {\"group\": \"host:p085\", \"fold\": 1}, \"p086\": {\"group\": \"host:p086\", \"fold\": 1}, \"p087\": {\"group\": \"host:p087\", \"fold\": 4}, \"p088\": {\"group\": \"host:p088\", \"fold\": 4}, \"p089\": {\"group\": \"host:p089\", \"fold\": 3}, \"p090\": {\"group\": \"host:p090\", \"fold\": 4}, \"p091\": {\"group\": \"host:p091\", \"fold\": 4}, \"p092\": {\"group\": \"host:p092\", \"fold\": 4}, \"p093\": {\"group\": \"host:p093\", \"fold\": 2}, \"p094\": {\"group\": \"host:p094\", \"fold\": 2}, \"p095\": {\"group\": \"host:p095\", \"fold\": 2}, \"p096\": {\"group\": \"host:p096\", \"fold\": 1}, \"p097\": {\"group\": \"host:p097\", \"fold\": 2}, \"p098\": {\"group\": \"host:p098\", \"fold\": 3}, \"p099\": {\"group\": \"host:p099\", \"fold\": 4}, \"p100\": {\"group\": \"host:p100\", \"fold\": 3}, \"p101\": {\"group\": \"host:p101\", \"fold\": 2}, \"p102\": {\"group\": \"host:p102\", \"fold\": 4}, \"p103\": {\"group\": \"host:p103\", \"fold\": 4}, \"p104\": {\"group\": \"host:p104\", \"fold\": 0}, \"p105\": {\"group\": \"host:p105\", \"fold\": 1}, \"p106\": {\"group\": \"host:p106\", \"fold\": 4}, \"p107\": {\"group\": \"host:p107\", \"fold\": 4}, \"p108\": {\"group\": \"host:p108\", \"fold\": 0}, \"p109\": {\"group\": \"host:p109\", \"fold\": 1}, \"p110\": {\"group\": \"host:p110\", \"fold\": 0}, \"p111\": {\"group\": \"host:p111\", \"fold\": 2}, \"p112\": {\"group\": \"host:p112\", \"fold\": 3}, \"p113\": {\"group\": \"host:p113\", \"fold\": 2}, \"p114\": {\"group\": \"host:p114\", \"fold\": 2}, \"p115\": {\"group\": \"host:p115\", \"fold\": 3}, \"p116\": {\"group\": \"host:p116\", \"fold\": 4}, \"p117\": {\"group\": \"host:p117\", \"fold\": 2}, \"p118\": {\"group\": \"host:p118\", \"fold\": 0}, \"p119\": {\"group\": \"host:p119\", \"fold\": 3}, \"p120\": {\"group\": \"host:p120\", \"fold\": 3}, \"p121\": {\"group\": \"host:p121\", \"fold\": 4}, \"p122\": {\"group\": \"host:p122\", \"fold\": 1}, \"p123\": {\"group\": \"host:p123\", \"fold\": 2}, \"p124\": {\"group\": \"host:p124\", \"fold\": 4}, \"p125\": {\"group\": \"host:p125\", \"fold\": 3}, \"p126\": {\"group\": \"host:p126\", \"fold\": 3}, \"p127\": {\"group\": \"host:p127\", \"fold\": 0}, \"p128\": {\"group\": \"host:p128\", \"fold\": 0}, \"p129\": {\"group\": \"host:p129\", \"fold\": 4}, \"p130\": {\"group\": \"host:p130\", \"fold\": 4}, \"p131\": {\"group\": \"host:p131\", \"fold\": 0}, \"p132\": {\"group\": \"host:p132\", \"fold\": 0}, \"p133\": {\"group\": \"host:p133\", \"fold\": 1}, \"p134\": {\"group\": \"host:p134\", \"fold\": 3}, \"p135\": {\"group\": \"host:p135\", \"fold\": 3}, \"p136\": {\"group\": \"host:p136\", \"fold\": 3}, \"p137\": {\"group\": \"host:p137\", \"fold\": 1}, \"p138\": {\"group\": \"host:p138\", \"fold\": 2}, \"p139\": {\"group\": \"host:p139\", \"fold\": 3}, \"p140\": {\"group\": \"host:p140\", \"fold\": 4}, \"p141\": {\"group\": \"host:p141\", \"fold\": 0}, \"p142\": {\"group\": \"host:p142\", \"fold\": 1}, \"p143\": {\"group\": \"host:p143\", \"fold\": 0}, \"p144\": {\"group\": \"host:p144\", \"fold\": 4}, \"p145\": {\"group\": \"host:p145\", \"fold\": 0}, \"p146\": {\"group\": \"host:p146\", \"fold\": 1}, \"p147\": {\"group\": \"host:p147\", \"fold\": 3}, \"p148\": {\"group\": \"host:p148\", \"fold\": 2}, \"p149\": {\"group\": \"host:p149\", \"fold\": 4}}, \"fold_sizes\": {\"1\": {\"training_rows\": 2159, \"validation_rows\": 540}, \"2\": {\"training_rows\": 2159, \"validation_rows\": 540}, \"3\": {\"training_rows\": 2160, \"validation_rows\": 539}, \"4\": {\"training_rows\": 2159, \"validation_rows\": 540}}, \"experiment\": \"improvement-v35\", \"protocol\": {\"experiment\": \"improvement-v35\", \"status\": \"frozen_before_training_not_launched\", \"hypothesis\": \"Matching negative examples on prompt ID may reduce topic shortcuts in author attribution.\", \"evidence_status\": \"Untested task-specific extension; SupCon establishes class-positive contrastive learning, not this prompt-control benefit.\", \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast/blob/master/losses.py\"], \"method_review\": \"Original SupCon averages positive log probabilities outside the log, excludes self comparisons, and operates on normalized projections. V18 loss retains that form with temperature/base-temperature ratio one. Original paper studies images; crossed text prompt sampling plus joint CE is our extension.\", \"primary_folds\": {\"file\": \"artifacts/baseline-v1/folds.csv\", \"sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"train\": [1, 2]}, \"comparison\": \"Both arms restore identical V18 4500-row external-only adapter, norm, head and projection. Same Qwen3-4B, tokenizer, host data, seed, two epochs, batch4, accumulation8, CE plus 0.1 SupCon, learning rates and fixed final V19 readout. Only host batch sampling changes.\", \"control\": \"Existing V18 independent prompt draws for each of two authors.\", \"candidate\": \"Two authors and two shared host prompt IDs form all four cells. Same-author positives span prompts; each anchor has a same-prompt different-author negative.\", \"external_groups\": \"Existing preparation constructs external groups from source + prompt_key. Cross-source prompt equivalence is not established. No external prompt matching or new external training is performed.\", \"scoring\": {\"metric\": \"Mean class recall, raw fixed V19 readout\", \"readout\": \"Layers 12/24/36, standardized logistic C=.02, mean log probabilities\", \"secondary\": \"Ordinary accuracy and existing fixed sparse blend may be reported, never used to replace primary gate.\"}, \"screen_gate\": {\"pooled_balanced_accuracy_gain_minimum\": 0.01, \"each_fold_gain_must_be_positive\": true, \"next_step\": \"Broader folds and full-incumbent ensemble evaluation; never automatic submission.\"}, \"runtime\": {\"wall_limit_seconds\": 5400, \"projection_seconds\": 3600, \"evidence\": \"Recorded V18 host stage about1483s plus V19 readout264s per fold. Two folds sequentially per arm, arms parallel on two T4 GPUs; estimate includes modest overhead but is not guaranteed.\", \"gpu_hours_estimate\": 2, \"execution\": \"One independent arm per GPU; terminate process groups at global deadline.\", \"paid_compute\": false, \"launch_requires\": \"Parent coordination, current quota and offline dependency preflight\", \"portable\": {\"execution\": \"Single GPU, independent then crossed; both fixed folds in each arm\", \"wall_limit_seconds\": 9000, \"projection_seconds\": 7000, \"setup_allowance_seconds\": 1200, \"setup_estimate_status\": \"Planning allowance, not measured; depends on image and download throughput\", \"cost_cap_usd\": 10, \"billing_guard\": \"Parent preflight must include setup, runtime and storage, and terminate paid pod after archive. Internal process timeout alone does not stop billing.\"}}, \"limitations\": [\"All host labels previously inspected; this is a screening comparison, not untouched confirmation.\", \"Sampling with replacement changes row exposure and negative difficulty; seed alone does not equate individual examples across arms.\", \"Single missing author/prompt cell handled by pairwise shared-prompt intersection, with slight sampling distribution change.\", \"No claim of improvement over .622 without full inference submission.\", \"Inference needs one text only, no prompt ID or other authors; same adapter architecture and cost as incumbent component.\"], \"attribution\": \"SupCon: Khosla et al.; Qwen model: Qwen team; public external pool: Nishant Dahal and original contributors. This experiment changes sampling only.\"}, \"protocol_sha256\": \"42a74562b98dce7ebc6cfa1ed4393c5584c426be9a14ccc0ad9987976a29b769\", \"arms\": {\"independent\": {\"parameters\": {\"experiment\": \"improvement-v18\", \"kind\": \"paired training-objective pilot\", \"model\": \"qwen-lm/qwen-3/Transformers/4b-base/1\", \"seed\": 1801, \"pilot_fold\": 0, \"external_cap_per_class\": 250, \"external_host_cosine_exclusion\": 0.3, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": [\"q_proj\", \"v_proj\"], \"pool_epochs\": 1, \"host_epochs\": 2, \"pool_adapter_lr\": 0.0001, \"pool_head_lr\": 0.0005, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"variants\": [\"ce\", \"ce_supcon\"], \"checkpoint_rule\": \"Evaluate fixed final host epoch only; no best-epoch selection\", \"promotion\": \"SupCon must improve final pilot accuracy by at least 0.01 over identical CE training before broader confirmation. Neither pilot alone qualifies for submission.\", \"wall_limit_seconds\": 5400, \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://arxiv.org/html/2106.09685v2\", \"https://github.com/microsoft/LoRA\", \"https://huggingface.co/docs/peft/package_reference/lora\"], \"attribution\": \"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0).\", \"limitations\": [\"One repeatedly inspected host fold, not untouched confirmation\", \"4500-row external cap screens objectives before scaling\", \"Small contrastive batches may limit the benefit\", \"Host-based external near-match exclusion uses unlabeled host text only for conservative leakage screening\"], \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"]}, \"variant\": \"ce_supcon\", \"readout_variant\": \"supcon_final\", \"training_worker\": \"independent_worker.py\", \"readout_worker\": \"independent_readout.py\", \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"checkpoint_sha256\": {\"heads.pt\": \"6c540f09084be67029f3651a020214fcb2afe53e95a127486985bf521a7e10ff\", \"adapter/adapter_model.safetensors\": \"353f3e3bacab0660d60b8b8e4a90e4d258f26fa5047a4010db3f2227b7bfb6d5\", \"adapter/README.md\": \"c7094bd33e036ce53b5d373d4c251d03e96d7b624f3ce89c9805e5f5844dd79b\", \"adapter/adapter_config.json\": \"5cc84f15952fcb3d9c192cc8d819fc201a31c73f95884656c5fe2acabda33ab7\"}, \"hidden_size\": 2560, \"origin_version\": 18, \"origin_worker_sha256\": \"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\", \"origin_readout_sha256\": \"c55bed7580eefde8f7461fff084510cd2c83fd5e0768cff066ae1ff700a4409c\"}, \"crossed\": {\"parameters\": {\"experiment\": \"improvement-v18\", \"kind\": \"paired training-objective pilot\", \"model\": \"qwen-lm/qwen-3/Transformers/4b-base/1\", \"seed\": 1801, \"pilot_fold\": 0, \"external_cap_per_class\": 250, \"external_host_cosine_exclusion\": 0.3, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": [\"q_proj\", \"v_proj\"], \"pool_epochs\": 1, \"host_epochs\": 2, \"pool_adapter_lr\": 0.0001, \"pool_head_lr\": 0.0005, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"variants\": [\"ce\", \"ce_supcon\"], \"checkpoint_rule\": \"Evaluate fixed final host epoch only; no best-epoch selection\", \"promotion\": \"SupCon must improve final pilot accuracy by at least 0.01 over identical CE training before broader confirmation. Neither pilot alone qualifies for submission.\", \"wall_limit_seconds\": 5400, \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://arxiv.org/html/2106.09685v2\", \"https://github.com/microsoft/LoRA\", \"https://huggingface.co/docs/peft/package_reference/lora\"], \"attribution\": \"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0).\", \"limitations\": [\"One repeatedly inspected host fold, not untouched confirmation\", \"4500-row external cap screens objectives before scaling\", \"Small contrastive batches may limit the benefit\", \"Host-based external near-match exclusion uses unlabeled host text only for conservative leakage screening\"], \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"]}, \"variant\": \"ce_supcon\", \"readout_variant\": \"supcon_final\", \"training_worker\": \"crossed_worker.py\", \"readout_worker\": \"crossed_readout.py\", \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"checkpoint_sha256\": {\"heads.pt\": \"6c540f09084be67029f3651a020214fcb2afe53e95a127486985bf521a7e10ff\", \"adapter/adapter_model.safetensors\": \"353f3e3bacab0660d60b8b8e4a90e4d258f26fa5047a4010db3f2227b7bfb6d5\", \"adapter/README.md\": \"c7094bd33e036ce53b5d373d4c251d03e96d7b624f3ce89c9805e5f5844dd79b\", \"adapter/adapter_config.json\": \"5cc84f15952fcb3d9c192cc8d819fc201a31c73f95884656c5fe2acabda33ab7\"}, \"hidden_size\": 2560, \"origin_version\": 18, \"origin_worker_sha256\": \"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\", \"origin_readout_sha256\": \"c55bed7580eefde8f7461fff084510cd2c83fd5e0768cff066ae1ff700a4409c\"}}, \"source_sha256\": {\"run_confirmation_arm_v35.py\": \"e8b210a1cd8528dc371bbdf4992bace3d68433681801afd0ebcf282732405a65\", \"prepare_likelihood_v33.py\": \"9517acff96c5f73c2838a3cc06bdc20aadeff9e14c801b460e3d6cd28c222838\", \"sampling_v35.py\": \"45a33b000a88593e25a9dc687b9918589cbc6ce263f168c4dfb4264cd2c73d1b\", \"independent_worker.py\": \"6d3371ddfef1c49d2a364c5e0b0f9152095c39d3c5302e2f8bdeb262ab16d96c\", \"independent_readout.py\": \"60ca39dfd35b1be99d73c0671776eee57feb193b480c20ba101c7d7761770fbb\", \"crossed_worker.py\": \"d99c55de2f32b5bfbdb9228aded62710309408d1ecc6f6c6e226a13cf8591796\", \"crossed_readout.py\": \"60ca39dfd35b1be99d73c0671776eee57feb193b480c20ba101c7d7761770fbb\"}, \"setup_sha256\": \"d9f5d28376d8467bb339cb16967911993afcf904e6828c83129c09b08a491b52\"}")

import torch
assert torch.cuda.device_count()==2
hosts=list(Path('/kaggle/input').rglob('train.csv'))
assert len(hosts)==1 and hashlib.sha256(hosts[0].read_bytes()).hexdigest()==plan['host_sha256']
plan['host_path']=str(hosts[0])
for arm,spec in plan['arms'].items():
    models=[]
    for p in Path('/kaggle/input').rglob('config.json'):
        cfg=json.loads(p.read_text())
        if cfg.get('model_type')=='qwen3' and cfg.get('hidden_size')==2560 and cfg.get('num_hidden_layers')==36:
            models.append(p.parent)
    assert len(models)==1,models
    spec['model_dir']=str(models[0])
    weights=[p for p in Path('/kaggle/input').rglob('adapter_model.safetensors')
             if p.parent.parent.name=='external_only' and p.parent.parent.parent.name=='ce_supcon'
             and hashlib.sha256(p.read_bytes()).hexdigest()==spec['checkpoint_sha256']['adapter/adapter_model.safetensors']]
    assert len(weights)==1
    spec['checkpoint']=str(weights[0].parent.parent)
    for relative,digest in spec['checkpoint_sha256'].items():
        assert hashlib.sha256((Path(spec['checkpoint'])/relative).read_bytes()).hexdigest()==digest
plan['deadline_unix']=time.time()+plan['protocol']['runtime']['wall_limit_seconds']
Path('plan.json').write_text(json.dumps(plan,indent=2))
processes=[]
try:
    for gpu,arm in enumerate(plan['arms']):
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu))
        processes.append(subprocess.Popen([sys.executable,'-u','run_confirmation_arm_v35.py',arm],env=env,start_new_session=True))
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):raise RuntimeError('V35 arm failed')
        if time.time()>plan['deadline_unix']:raise TimeoutError('V35 hard deadline')
        time.sleep(10)
    assert all(p.returncode==0 for p in processes)
finally:
    for p in processes:
        try: os.killpg(p.pid,signal.SIGTERM)
        except ProcessLookupError: pass
    time.sleep(2)
    for p in processes:
        try: os.killpg(p.pid,signal.SIGKILL)
        except ProcessLookupError: pass
    for p in processes:p.wait()
Path('summary.json').write_text(json.dumps({'completed':True,'experiment':'improvement-v35','submission_authorized':False}))
